#  Category Tree - Bronze Ingestion


## Imports

In [0]:
import uuid
from pyspark.sql.functions import col, current_timestamp, lit
from pyspark.sql.types import StringType, StructField, StructType

## Configuration

In [0]:
environment = "dev"

catalog = f"ecommerce_{environment}"
schema = "bronze"
table_name = "retailrocket_category_tree"
target_table = f"{catalog}.{schema}.{table_name}"

source_system = "retailrocket"
source_dataset = "category_tree"

source_path = (
    f"/Volumes/{catalog}/landing/raw_files/"
    f"{source_system}/{source_dataset}/"
)

checkpoint_path = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/checkpoint/"
)

schema_location = (
    f"/Volumes/{catalog}/ops/runtime_state/"
    f"autoloader/{source_system}/{source_dataset}/schema/"
)

run_id = str(uuid.uuid4())

## Schema Definition

In [0]:
source_schema = StructType([
    StructField("categoryid", StringType(), True),
    StructField("parentid", StringType(), True)
])

## Read with Auto Loader

In [0]:
source_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.schemaLocation", schema_location)
    .option("cloudFiles.schemaEvolutionMode", "rescue")
    .option("rescuedDataColumn", "_rescued_data")
    .option("header", "true")
    .schema(source_schema)
    .load(source_path)
)

## Add Bronze metadata

In [0]:
bronze_df = (
    source_df
    .withColumn("source_file_path", col("_metadata.file_path"))
    .withColumn("source_file_modification_time", col("_metadata.file_modification_time"))
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("ingestion_run_id", lit(run_id))
    .withColumn("source_system", lit(source_system))
    .withColumn("source_dataset", lit(source_dataset))
)

## Write to the Bronze table

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable(target_table)
)

query.awaitTermination()

In [0]:
spark.table(target_table).printSchema()

root
 |-- categoryid: string (nullable = true)
 |-- parentid: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- source_file_path: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- ingestion_run_id: string (nullable = true)
 |-- source_system: string (nullable = true)
 |-- source_dataset: string (nullable = true)



In [0]:
display(spark.table(target_table).limit(5))

categoryid,parentid,_rescued_data,source_file_path,source_file_modification_time,ingestion_timestamp,ingestion_run_id,source_system,source_dataset
1016,213,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
809,169,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
570,9,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
1691,885,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree
536,1691,null,/Volumes/ecommerce_dev/landing/raw_files/retailrocket/category_tree/category_tree.csv,2026-08-02T21:33:23.000Z,2026-08-03T03:41:37.019Z,ca28b80f-1f6e-44e3-b363-955abae8c1b6,retailrocket,category_tree


In [0]:
spark.table(target_table).count()

1669